# BayesNF — baseline (VI + MAP)

Канонический baseline для BayesNF на данных ReKIS. Две parallel-фиксации:
**VI** (`BayesianNeuralFieldVI`) и **MAP** (`BayesianNeuralFieldMAP`). Один и тот
же набор фич и сезонность по умолчанию; гиперпараметры — в ячейках, меняешь руками.

После каждого `train_eval_*` артефакты автоматически летят в S3 в структурированную
папку по `name=...`:

```
s3://thesis-data-ismaktam/bayesnf/runs/<name>/
  config.json   metrics.json   preds.parquet   losses.npy   model.pkl
```

Повторный запуск с тем же `name` перезаписывает папку.

Метрики расширены: явное разделение `dry / wet / wet-detector` с порогами
`observed ≥ 0.5 mm` и `predicted ≥ 0.4 mm`. Аналитика калибровки, spread–skill
и семивариограммы считается на **wet-only** подмножестве, потому что глобальная
выглядит обманчиво из-за массы нулей.


## 1. Окружение + импорты


In [ ]:
import os
# JAX env vars ДОЛЖНЫ быть установлены ДО первого `import jax`
os.environ.setdefault('JAX_LOG_COMPILES', '0')
os.environ.setdefault('XLA_PYTHON_CLIENT_MEM_FRACTION', '0.90')

# Speedup flags (validated in speedup/speedup.ipynb): включают Triton GEMM,
# латентность-скрывающий шедулер и async collectives. На smoke A/B давали
# одинаковую численность с tf32 (loss совпадает до 5-й цифры).
os.environ['XLA_FLAGS'] = (
    '--xla_gpu_enable_triton_gemm=true '
    '--xla_gpu_enable_latency_hiding_scheduler=true '
    '--xla_gpu_enable_async_collectives=true'
)

import sys
import gc
import time
import json
import pickle
import threading
from pathlib import Path

import numpy as np
import pandas as pd

import jax
import jax.numpy as jnp
import bayesnf
from bayesnf.spatiotemporal import (
    BayesianNeuralFieldVI,
    BayesianNeuralFieldMAP,
)

# bf16 mixed precision — на A100 включает tensor cores на полной полосе.
# В smoke A/B (notebooks/05_bayesnf/speedup/) ELBO остался finite, лосс
# совпал с tf32 до 5-й значащей цифры → численно безопасно.
# Главный профит — ~½ peak memory на активациях, что позволяет держать
# batch=131072 даже на длинных периодах (>10 лет).
jax.config.update('jax_default_matmul_precision', 'bfloat16')
PRECISION_NOTE = 'bfloat16 (A100 tensor cores)'

# Repo root: notebooks/05_bayesnf/baseline/<этот>.ipynb → ../../..
ROOT = Path('../../..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)

print(f'cwd      : {os.getcwd()}')
print(f'bayesnf  : {getattr(bayesnf, "__version__", "n/a")}')
print(f'precision: {PRECISION_NOTE}')


## 2. GPU-диагностика


In [ ]:
import subprocess

backend = jax.default_backend()
devices = jax.devices()
print(f'JAX backend : {backend}')
print(f'JAX devices : {devices}')
print(f'n local     : {jax.local_device_count()}')

if backend != 'gpu':
    print('\n⚠ JAX на CPU — этот ноутбук рассчитан на GPU.')

try:
    smi = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.used,memory.total',
         '--format=csv,noheader'],
        capture_output=True, text=True, timeout=5,
    )
    print('\nnvidia-smi:')
    print(smi.stdout)
except Exception as e:
    print(f'nvidia-smi unavailable: {e}')


## 3. Конфиг данных и таргета

Likelihood / target / период данных. Гиперпараметры модели — в ячейках train_eval ниже.


In [ ]:
# --- Likelihood и target ---
LIKELIHOOD    = 'ZINB'
TARGET_COL    = 'rainfall_int'
NEEDS_RESCALE = True
PRECIP_SCALE  = 10

# --- Период данных ---
FOLD       = 0
YEAR_START = 2013
YEAR_END   = 2022

# --- Output ---
LOCAL_RUNS = Path('results/bayesnf/runs')
LOCAL_RUNS.mkdir(parents=True, exist_ok=True)

# --- S3 ---
S3_BUCKET    = 'thesis-data-ismaktam'
S3_RUNS_ROOT = 'bayesnf/runs'

print(f'likelihood   : {LIKELIHOOD}  (target={TARGET_COL}, rescale={NEEDS_RESCALE})')
print(f'years        : {YEAR_START}-{YEAR_END}, fold {FOLD}')
print(f'local runs   : {LOCAL_RUNS}')
print(f's3 prefix    : s3://{S3_BUCKET}/{S3_RUNS_ROOT}/<name>/')


## 4. Загрузка данных из S3

Фичи (включая FFRK: `idw, gos, svd_05..09`) уже посчитаны и лежат в
`s3://thesis-data-ismaktam/bayesnf/data/`. Грузим все потенциально нужные колонки —
конкретный subset выберется через `feature_cols` в train_eval.


In [ ]:
DATA_S3_DIR = 'bayesnf/data'
DATA_DIR    = Path('results/bayesnf/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

train_path = DATA_DIR / f'bayesnf_fold{FOLD}_train.parquet'
test_path  = DATA_DIR / f'bayesnf_fold{FOLD}_test.parquet'

def _ensure_local(local_path: Path, s3_key: str) -> None:
    if local_path.exists():
        print(f'  cache hit: {local_path.name} ({local_path.stat().st_size/1e6:.0f} MB)')
        return
    import boto3
    print(f'  downloading s3://{S3_BUCKET}/{s3_key} ...')
    boto3.client('s3').download_file(S3_BUCKET, s3_key, str(local_path))
    print(f'  ↓ {local_path.name} ({local_path.stat().st_size/1e6:.0f} MB)')

_ensure_local(train_path, f'{DATA_S3_DIR}/bayesnf_fold{FOLD}_train.parquet')
_ensure_local(test_path,  f'{DATA_S3_DIR}/bayesnf_fold{FOLD}_test.parquet')

date_filter = [
    ('datetime', '>=', pd.Timestamp(f'{YEAR_START}-01-01')),
    ('datetime', '<=', pd.Timestamp(f'{YEAR_END}-12-31')),
]

ALL_GEO_COLS = [
    'datetime', 'latitude', 'longitude', 'x_proj', 'y_proj', 'elevation_m',
    'idw', 'gos',
    'svd_05', 'svd_06', 'svd_07', 'svd_08', 'svd_09',
    'bulk_density', 'clay', 'sand', 'silt', 'soc', 'water_10kpa',
]
KEEP_COLS = ALL_GEO_COLS + ['rainfall', 'rainfall_int', 'station_id']

df_train = pd.read_parquet(train_path, filters=date_filter)[KEEP_COLS].copy()
df_test  = pd.read_parquet(test_path,  filters=date_filter)[KEEP_COLS].copy()
df_train['datetime'] = pd.to_datetime(df_train['datetime'])
df_test ['datetime'] = pd.to_datetime(df_test ['datetime'])

assert df_train[ALL_GEO_COLS].isna().sum().sum() == 0, 'NaN in geo cols'
assert df_test [ALL_GEO_COLS].isna().sum().sum() == 0
assert 'rainfall_int' in df_train.columns, 'ZINB needs rainfall_int = round(10·mm)'

print(f'train rows / stations : {len(df_train):>10,} / {df_train["station_id"].nunique()}')
print(f'test  rows / stations : {len(df_test):>10,} / {df_test ["station_id"].nunique()}')
print(f'dates                 : {df_train["datetime"].min().date()} → {df_train["datetime"].max().date()}')
print(f'rainfall (mm) range   : [{df_train["rainfall"].min():.2f}, {df_train["rainfall"].max():.2f}]')
print(f'%zero (rainfall_int)  : {100*(df_train["rainfall_int"]==0).mean():.1f}%')
df_train.head()


## 5. JIT-cache patch для `predict`

Тот же патч что в time_seasonality / geo_features. Без него `predict_bnf`
перекомпилирует closure на каждый chunk.


In [ ]:
import bayesnf.inference as bnf_inference
import bayesnf.models as bnf_models
import bayesnf.spatiotemporal as bnf_st
from tensorflow_probability.substrates import jax as _tfp_jax

_tfd = _tfp_jax.distributions
_LD  = bnf_models.LikelihoodDist

# --- bounded likelihood (anti-blowup для NB / ZINB) ---------------------------
# NB/ZINB параметризация в bayesnf: mean = softplus(MLP(x)), shape = softplus(p1).
# MLP-выход не ограничен сверху → на тестовых строках с экстремальными фичами
# `mean` уходит в ~10^6 mm. Защита: тонкая tanh-обёртка вокруг сырых параметров.
#
#  raw_pred  → PRED_CAP * tanh(raw_pred / PRED_CAP)   (мягкий клип; в линейном
#              режиме при |raw_pred| << PRED_CAP не мешает обучению).
#  params[1] → SHAPE_CAP * tanh(params[1] / SHAPE_CAP) → shape ∈ ~[2.5e-3, 6]
#              → total_count = 1/shape ∈ ~[0.17, 400].
#
# Цель — диапазон mean в rainfall_int единицах: target ≤ 10·200 mm = 2000.
# PRED_CAP = 4000 (с двойным запасом) → softplus(±4000) ≈ {≈0, 4000}.
PRED_CAP  = 4000.0
SHAPE_CAP = 6.0

_orig_make_likelihood = bnf_models.make_likelihood_model


def make_likelihood_bounded(params, x, mlp, mlp_template, distribution):
    if _LD(distribution) == _LD.NORMAL:
        return _orig_make_likelihood(params, x, mlp, mlp_template, distribution)

    treedef    = jax.tree_util.tree_structure(mlp_template)
    mlp_params = jax.tree_util.tree_unflatten(treedef, params[3:])
    raw_pred   = mlp.apply(mlp_params, x)
    bounded_pred = PRED_CAP * jnp.tanh(raw_pred / PRED_CAP)
    mean       = jax.nn.softplus(bounded_pred)

    bounded_p1 = SHAPE_CAP * jnp.tanh(params[1] / SHAPE_CAP)
    shape      = jax.nn.softplus(bounded_p1)
    logits     = -jnp.log(shape) - jnp.log(mean)

    if _LD(distribution) == _LD.NB:
        return _tfd.Independent(
            _tfd.NegativeBinomial(total_count=1.0 / shape, logits=logits), 1)
    elif _LD(distribution) == _LD.ZINB:
        inflated_loc_probs = 1.0 / (1.0 + jnp.exp(-params[2]))
        return _tfd.Independent(
            _tfd.ZeroInflatedNegativeBinomial(
                total_count=1.0 / shape, logits=logits,
                inflated_loc_probs=inflated_loc_probs * jnp.ones(shape=mean.shape)), 1)
    raise AssertionError(f'unknown distribution: {distribution}')


bnf_models.make_likelihood_model = make_likelihood_bounded
print(f'✓ bounded likelihood: mean ≤ ~{PRED_CAP:.0f} (rainfall_int units = '
      f'{PRED_CAP/PRECIP_SCALE:.0f} mm/day),  total_count ∈ [{1/jax.nn.softplus(SHAPE_CAP):.3f}, '
      f'{1/jax.nn.softplus(-SHAPE_CAP):.1f}]')


# --- predict JIT-cache (тот же что в time_seasonality / geo_features) --------
def _patched_predict(self, table, quantiles=(0.5,), approximate_quantiles=False):
    test_data = self.data_handler.get_test(table)
    distribution = bnf_models.LikelihoodDist(self.observation_model)

    if getattr(self, '_cached_forecast_inner', None) is None:
        model_args = self._model_args(test_data.shape)
        fn = bnf_inference._make_forecast_inner(model_args, distribution)
        for _ in range(self._ensemble_dims - 1):
            fn = jax.vmap(fn, in_axes=(0, None))
        fn = jax.pmap(fn, in_axes=(0, None))
        self._cached_forecast_inner = fn

    forecast_inner = self._cached_forecast_inner
    axis = tuple(range(self._ensemble_dims))

    forecast_params = bnf_inference.forecast_parameters_batched(
        test_data, self.params_, distribution, forecast_inner,
    )

    if distribution == bnf_models.LikelihoodDist.NORMAL:
        means, scales = forecast_params
        forecast_means = means
        forecast_quantiles = bnf_inference._get_percentile_normal(
            forecast_means, scales, quantiles, axis=axis,
            approximate=approximate_quantiles,
        )
    elif distribution in (bnf_models.LikelihoodDist.NB, bnf_models.LikelihoodDist.ZINB):
        obs_d = bnf_inference._build_observation_distribution(distribution, forecast_params)
        forecast_means = obs_d.mean()
        forecast_quantiles = jax.vmap(
            lambda q: bnf_inference._get_nb_quantiles_root(obs_d, q, ensemble_axes=axis)
        )(jnp.array(quantiles))
    else:
        raise ValueError(f'Unknown distribution: {distribution}')

    return forecast_means, forecast_quantiles


bnf_st.BayesianNeuralFieldEstimator.predict = _patched_predict
print('✓ patched BayesianNeuralFieldEstimator.predict')


## 6. Helpers: метрики, сохранение, S3-upload, train_eval_vi / train_eval_map

Один большой блок. Запускается один раз; ниже две тренировочные ячейки просто
зовут `train_eval_vi(name=..., **overrides)` или `train_eval_map(...)`.

**Что считают метрики.**

| Группа | Слайс | Метрики |
|--------|-------|---------|
| Global | весь test | RMSE, MAE, bias, CRPS, cov90, cov80 |
| Wet    | `y ≥ 0.5 mm` | RMSE_wet, MAE_wet, CRPS_wet, cov90_wet, cov80_wet |
| Dry    | `y < 0.5 mm` | MAE_dry, RMSE_dry, bias_dry |
| Detector | wet-classifier с порогом `predicted ≥ 0.4 mm`, observed `≥ 0.5 mm` | TP/FP/TN/FN, accuracy, precision, recall, F1, Brier |


In [ ]:
from properscoring import crps_ensemble
import boto3

# --- thresholds ---
WET_THRESHOLD_MM   = 0.5   # y_true wet ⇔ y ≥ 0.5 mm
PRED_DRY_THRESHOLD = 0.4   # модель «предсказала dry» ⇔ mean_mm < 0.4

# --- quantiles ---
QUANTILES = (0.05, 0.1, 0.2, 0.3, 0.4, 0.50, 0.6, 0.7, 0.8, 0.9, 0.95)
QUANTILE_LABELS = [5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 95]
Q05_I = QUANTILE_LABELS.index(5)
Q10_I = QUANTILE_LABELS.index(10)
Q50_I = QUANTILE_LABELS.index(50)
Q90_I = QUANTILE_LABELS.index(90)
Q95_I = QUANTILE_LABELS.index(95)

PRED_CHUNK = 500_000

# --- defaults (geo features + seasonality) ---
GEO_DEFAULTS = dict(
    feature_cols=['datetime', 'latitude', 'longitude', 'elevation_m',
                  'idw', 'gos',
                  'svd_05', 'svd_06', 'svd_07', 'svd_08', 'svd_09',
                  'bulk_density', 'clay', 'sand', 'silt', 'soc', 'water_10kpa'],
    interactions=[(0, 1), (0, 2), (0, 3), (0, 4), (0, 5),
                  (1, 2), (1, 3), (2, 3)],
)
SEAS_DEFAULT = dict(periods=['W', 'Y'], harmonics=[1, 10])


# ──────────────────────────────────────────────────────────────────────
def _metrics(y_true, mean_mm, q_arr):
    """Returns dict with global / wet / dry / detector groups."""
    lo90 = q_arr[:, Q05_I]; hi90 = q_arr[:, Q95_I]
    lo80 = q_arr[:, Q10_I]; hi80 = q_arr[:, Q90_I]

    obs_wet  = y_true  >= WET_THRESHOLD_MM
    pred_wet = mean_mm >= PRED_DRY_THRESHOLD
    tp = int(( obs_wet &  pred_wet).sum())
    tn = int((~obs_wet & ~pred_wet).sum())
    fp = int((~obs_wet &  pred_wet).sum())
    fn = int(( obs_wet & ~pred_wet).sum())
    n  = len(y_true)
    eps = 1e-12

    accuracy  = (tp + tn) / max(n, 1)
    precision = tp / max(tp + fp, 1)
    recall    = tp / max(tp + fn, 1)
    f1        = 2 * precision * recall / max(precision + recall, eps)
    brier     = float(np.mean((pred_wet.astype(np.float32) - obs_wet.astype(np.float32))**2))

    out = {
        # global
        'rmse'    : float(np.sqrt(np.mean((mean_mm - y_true) ** 2))),
        'mae'     : float(np.mean(np.abs(mean_mm - y_true))),
        'bias'    : float(np.mean(mean_mm - y_true)),
        'crps'    : float(crps_ensemble(y_true, q_arr).mean()),
        'cov90'   : float(np.mean((y_true >= lo90) & (y_true <= hi90))),
        'cov80'   : float(np.mean((y_true >= lo80) & (y_true <= hi80))),
        'n_total' : int(n),

        # detector
        'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn,
        'accuracy' : float(accuracy),
        'precision': float(precision),
        'recall'   : float(recall),
        'f1'       : float(f1),
        'brier'    : brier,
    }

    # wet slice
    if obs_wet.sum() > 0:
        out.update({
            'rmse_wet'  : float(np.sqrt(np.mean((mean_mm[obs_wet] - y_true[obs_wet]) ** 2))),
            'mae_wet'   : float(np.mean(np.abs(mean_mm[obs_wet] - y_true[obs_wet]))),
            'crps_wet'  : float(crps_ensemble(y_true[obs_wet], q_arr[obs_wet]).mean()),
            'cov90_wet' : float(np.mean((y_true[obs_wet] >= lo90[obs_wet]) & (y_true[obs_wet] <= hi90[obs_wet]))),
            'cov80_wet' : float(np.mean((y_true[obs_wet] >= lo80[obs_wet]) & (y_true[obs_wet] <= hi80[obs_wet]))),
            'n_wet'     : int(obs_wet.sum()),
        })
    else:
        out.update({k: float('nan') for k in
                    ('rmse_wet','mae_wet','crps_wet','cov90_wet','cov80_wet')})
        out['n_wet'] = 0

    # dry slice
    if (~obs_wet).sum() > 0:
        d = ~obs_wet
        out.update({
            'mae_dry'  : float(np.mean(np.abs(mean_mm[d] - y_true[d]))),
            'rmse_dry' : float(np.sqrt(np.mean((mean_mm[d] - y_true[d]) ** 2))),
            'bias_dry' : float(np.mean(mean_mm[d] - y_true[d])),
            'n_dry'    : int(d.sum()),
        })
    else:
        out.update({'mae_dry': float('nan'), 'rmse_dry': float('nan'),
                    'bias_dry': float('nan'), 'n_dry': 0})
    return out


def _save_run(run_name: str, model, losses, preds_df, metrics, config):
    """Persist all artifacts under LOCAL_RUNS/<run_name>/.

    Известная TFP-проблема: `model.params_` — это `structural_tuple.StructTuple`,
    класс создаётся динамически внутри функции (`weakref` кэш), `pickle` не
    может найти его по qualified-name при загрузке. Поэтому раскладываем в dict
    с явными именами полей и сохраняем как обычные numpy-листья.
    """
    sub_dir = LOCAL_RUNS / run_name
    sub_dir.mkdir(parents=True, exist_ok=True)

    np.save(sub_dir / 'losses.npy', np.asarray(losses))
    with open(sub_dir / 'metrics.json', 'w') as f:
        json.dump(metrics, f, indent=2)
    with open(sub_dir / 'config.json', 'w') as f:
        json.dump(config, f, indent=2)
    preds_df.to_parquet(sub_dir / 'preds.parquet', index=False)

    # Разрушаем StructTuple → плоский dict.
    p = model.params_
    params_dict = {name: np.asarray(val) for name, val in zip(p._fields, p)}

    # Все аргументы конструктора, чтобы можно было пересоздать модель на load.
    init_kwargs = {
        'width': model.width, 'depth': model.depth,
        'freq': model.freq, 'timetype': model.timetype,
        'seasonality_periods': list(model.seasonality_periods),
        'num_seasonal_harmonics': list(model.num_seasonal_harmonics),
        'feature_cols': list(model.feature_cols),
        'target_col':  model.target_col,
        'standardize': list(model.standardize),
        'observation_model': model.observation_model,
        'interactions': [list(t) for t in model.interactions]
                         if model.interactions is not None else None,
    }

    # data_handler-овые скейлеры (mean / std для standardize) — нужны при predict.
    dh = model.data_handler
    standardize_ = {
        'mean': np.asarray(dh.standardize_['mean'])  if hasattr(dh, 'standardize_') and dh.standardize_ else None,
        'std' : np.asarray(dh.standardize_['std'])   if hasattr(dh, 'standardize_') and dh.standardize_ else None,
    }

    payload = {
        'params_dict' : params_dict,
        'param_fields': list(p._fields),
        'init_kwargs' : init_kwargs,
        'standardize_': standardize_,
        'inference'   : config.get('inference'),
    }
    with open(sub_dir / 'model.pkl', 'wb') as f:
        pickle.dump(payload, f)
    print(f'  ✓ saved {len(params_dict)} param fields + meta to {sub_dir / "model.pkl"}')
    return sub_dir


def _load_run_model(run_name: str):
    """Симметрия к _save_run: вернуть инициализированный estimator готовый к predict.

    Не обязательно нужен в baseline-ноутбуке (он читает preds.parquet напрямую),
    но пригодится для grid-предикций позже.
    """
    sub_dir = LOCAL_RUNS / run_name
    with open(sub_dir / 'model.pkl', 'rb') as f:
        payload = pickle.load(f)

    from tensorflow_probability.python.internal import structural_tuple
    inference = payload['inference']
    cls = bnf_st.BayesianNeuralFieldVI if inference == 'vi' else bnf_st.BayesianNeuralFieldMAP
    model = cls(**payload['init_kwargs'])

    # Пересобираем StructTuple из dict
    StructT = structural_tuple.structtuple(payload['param_fields'])
    model.params_ = StructT(*(payload['params_dict'][k] for k in payload['param_fields']))

    # Восстановить скейлеры
    std = payload.get('standardize_') or {}
    if std.get('mean') is not None and std.get('std') is not None:
        model.data_handler.standardize_ = {
            'mean': std['mean'], 'std': std['std'],
        }
    return model


def _upload_run_s3(run_name: str) -> None:
    """Idempotently upload LOCAL_RUNS/<run_name>/ to s3://bucket/bayesnf/runs/<run_name>/."""
    src_dir = LOCAL_RUNS / run_name
    if not src_dir.exists():
        print(f'  ⚠ {src_dir} does not exist, skip upload')
        return
    s3 = boto3.client('s3')
    for f in sorted(src_dir.iterdir()):
        if not f.is_file():
            continue
        key = f'{S3_RUNS_ROOT}/{run_name}/{f.name}'
        s3.upload_file(str(f), S3_BUCKET, key)
        size_mb = f.stat().st_size / 1e6
        print(f'  ↑ s3://{S3_BUCKET}/{key}  ({size_mb:.1f} MB)')


# ──────────────────────────────────────────────────────────────────────
def _build_predict_metrics(model, df_test_sub, name):
    """Run chunked predict, compute metrics, return (mean_per_point, q_arr, metrics_dict, preds_df, predict_time_s)."""
    means_chunks, q_chunks = [], []
    t0 = time.time()
    n = len(df_test_sub)
    for i in range(0, n, PRED_CHUNK):
        j = min(i + PRED_CHUNK, n)
        m, q = model.predict(df_test_sub.iloc[i:j], quantiles=QUANTILES, approximate_quantiles=True)
        means_chunks.append(np.asarray(m))
        q_chunks.append(np.asarray(q))
    gc.collect()
    predict_time_s = time.time() - t0
    print(f'  ✓ predict {n:,} rows in {predict_time_s:.0f}s')

    means_raw = np.concatenate(means_chunks, axis=-1)
    q_raw     = np.concatenate(q_chunks, axis=-1)
    mean_per_point = means_raw.reshape(-1, n).mean(axis=0)
    q_flat = q_raw.reshape(q_raw.shape[0], -1, n).mean(axis=1)

    if NEEDS_RESCALE:
        mean_per_point = mean_per_point / PRECIP_SCALE
        q_flat = q_flat / PRECIP_SCALE

    # --- diagnostics + safety clip ---------------------------------------
    n_bad_mean = int((~np.isfinite(mean_per_point)).sum())
    n_bad_q    = int((~np.isfinite(q_flat)).sum())
    abs_mean   = np.abs(mean_per_point)
    pct = np.nanpercentile(abs_mean, [50, 95, 99, 99.9])
    print(f'  diag |mean_mm|: p50={pct[0]:.2f}  p95={pct[1]:.2f}  '
          f'p99={pct[2]:.2f}  p99.9={pct[3]:.2f}  max={np.nanmax(abs_mean):.2f}')
    print(f'  diag non-finite: mean={n_bad_mean} ({n_bad_mean/n*100:.3f}%) '
          f'q={n_bad_q} ({n_bad_q/q_flat.size*100:.3f}%)')

    # Сейф-нет клип: 250 mm/day абсолютный максимум на станционных данных
    # (мировой суточный рекорд ≈ 1825 mm — это редчайший случай; в Германии
    # реалистично ≤ 200 mm). Клипим mean агрессивнее чем квантили — не портим
    # калибровку.
    HARD_CAP_MEAN_MM = 250.0
    HARD_CAP_Q_MM    = 500.0
    n_clip_mean = int((mean_per_point > HARD_CAP_MEAN_MM).sum())
    n_clip_q    = int((q_flat > HARD_CAP_Q_MM).sum())
    if n_clip_mean or n_clip_q:
        print(f'  ⚠ clipping: mean>{HARD_CAP_MEAN_MM}mm n={n_clip_mean}, '
              f'q>{HARD_CAP_Q_MM}mm n={n_clip_q}')
    mean_per_point = np.clip(np.nan_to_num(mean_per_point, nan=0.0,
                                           posinf=HARD_CAP_MEAN_MM, neginf=0.0),
                             0.0, HARD_CAP_MEAN_MM)
    q_flat = np.clip(np.nan_to_num(q_flat, nan=0.0,
                                    posinf=HARD_CAP_Q_MM, neginf=0.0),
                     0.0, HARD_CAP_Q_MM)

    y_true = df_test_sub['rainfall'].to_numpy()
    q_arr = np.stack(list(q_flat), axis=1)
    q_arr = np.sort(q_arr, axis=1)

    metrics = _metrics(y_true, mean_per_point, q_arr)
    metrics.update({
        'name': name, 'n_quantiles': len(QUANTILES),
        'predict_time_s': float(predict_time_s),
        'n_nonfinite_mean': n_bad_mean,
        'n_nonfinite_q'   : n_bad_q,
        'n_clipped_mean'  : n_clip_mean,
        'n_clipped_q'     : n_clip_q,
        'p99_abs_mean_mm' : float(pct[2]),
        'p999_abs_mean_mm': float(pct[3]),
    })

    preds_df = df_test_sub[['station_id', 'datetime']].copy()
    preds_df['observed_mm'] = y_true
    preds_df['mean_mm']     = mean_per_point
    for lbl, qv in zip(QUANTILE_LABELS, q_flat):
        preds_df[f'q{lbl:02d}'] = qv

    return mean_per_point, q_arr, metrics, preds_df


def _print_summary(metrics):
    m = metrics
    print(f'  CRPS     = {m["crps"]:.4f}    CRPS_wet = {m["crps_wet"]:.4f}')
    print(f'  MAE      = {m["mae"]:.4f}    MAE_wet  = {m["mae_wet"]:.4f}    MAE_dry  = {m["mae_dry"]:.4f}')
    print(f'  RMSE     = {m["rmse"]:.4f}   RMSE_wet = {m["rmse_wet"]:.4f}   RMSE_dry = {m["rmse_dry"]:.4f}')
    print(f'  bias     = {m["bias"]:+.4f}  cov90    = {m["cov90"]:.3f}      cov80    = {m["cov80"]:.3f}')
    print(f'  detector → acc={m["accuracy"]:.3f}  P={m["precision"]:.3f}  R={m["recall"]:.3f}  F1={m["f1"]:.3f}  Brier={m["brier"]:.4f}')
    print(f'              TP={m["tp"]:,}  FP={m["fp"]:,}  TN={m["tn"]:,}  FN={m["fn"]:,}')


def _heartbeat_start(name):
    stop = threading.Event()
    def _beat():
        t0 = time.time()
        while not stop.wait(20.0):
            print(f'    [{name}] still training ({time.time()-t0:.0f}s)', flush=True)
    thr = threading.Thread(target=_beat, daemon=True); thr.start()
    return stop, thr


# ──────────────────────────────────────────────────────────────────────
def train_eval_vi(
    name: str,
    *,
    num_epochs: int,
    batch_size: int,
    ensemble_size: int,
    learning_rate: float,
    kl_weight: float,
    sample_size_posterior: int,
    width: int = 256,
    depth: int = 2,
    seed: int = 0,
    seasonality_periods=None,
    num_seasonal_harmonics=None,
    feature_cols=None,
    interactions=None,
) -> dict:
    seasonality_periods    = seasonality_periods    or SEAS_DEFAULT['periods']
    num_seasonal_harmonics = num_seasonal_harmonics or SEAS_DEFAULT['harmonics']
    feature_cols           = feature_cols           or GEO_DEFAULTS['feature_cols']
    interactions           = interactions           or GEO_DEFAULTS['interactions']

    assert len(seasonality_periods) == len(num_seasonal_harmonics)
    assert feature_cols[0] == 'datetime'
    for i, j in interactions:
        assert 0 <= i < len(feature_cols) and 0 <= j < len(feature_cols)

    print(f'\n{"="*78}')
    print(f'  [VI] {name}')
    print(f'       seasonality={seasonality_periods}/{num_seasonal_harmonics}')
    print(f'       features ({len(feature_cols)}): {feature_cols}')
    print(f'       width={width} depth={depth}  epochs={num_epochs}  batch={batch_size}')
    print(f'       ensemble={ensemble_size}  lr={learning_rate}  kl={kl_weight}  sample_post={sample_size_posterior}')
    print(f'{"="*78}')

    standardize_cols = [c for c in feature_cols if c != 'datetime']
    model = BayesianNeuralFieldVI(
        width=width, depth=depth,
        freq='D', timetype='index',
        seasonality_periods=seasonality_periods,
        num_seasonal_harmonics=num_seasonal_harmonics,
        feature_cols=feature_cols,
        target_col=TARGET_COL,
        standardize=standardize_cols,
        observation_model=LIKELIHOOD,
        interactions=interactions,
    )

    keep = feature_cols + ['rainfall', 'rainfall_int', 'station_id']
    df_train_sub = df_train[keep]
    df_test_sub  = df_test [keep]

    stop, thr = _heartbeat_start(name)
    t0 = time.time()
    try:
        model = model.fit(
            df_train_sub,
            seed=jax.random.PRNGKey(seed),
            num_epochs=num_epochs,
            ensemble_size=ensemble_size,
            learning_rate=learning_rate,
            batch_size=batch_size,
            kl_weight=kl_weight,
            sample_size_posterior=sample_size_posterior,
        )
    finally:
        stop.set(); thr.join(timeout=1)
    fit_time_s = time.time() - t0
    print(f'  ✓ fit done in {fit_time_s:.0f}s ({fit_time_s/60:.1f} min)')

    losses_arr = np.asarray(model.losses_)
    _, _, metrics, preds_df = _build_predict_metrics(model, df_test_sub, name)
    metrics.update({
        'inference'             : 'vi',
        'seasonality_periods'   : seasonality_periods,
        'num_seasonal_harmonics': num_seasonal_harmonics,
        'feature_cols'          : feature_cols,
        'interactions'          : interactions,
        'n_features'            : len(feature_cols),
        'fit_time_s'            : float(fit_time_s),
    })

    config = {
        'inference': 'vi', 'name': name,
        'width': width, 'depth': depth,
        'num_epochs': num_epochs, 'batch_size': batch_size,
        'ensemble_size': ensemble_size, 'learning_rate': learning_rate,
        'kl_weight': kl_weight, 'sample_size_posterior': sample_size_posterior,
        'seed': seed,
        'seasonality_periods': seasonality_periods,
        'num_seasonal_harmonics': num_seasonal_harmonics,
        'feature_cols': feature_cols, 'interactions': interactions,
        'likelihood': LIKELIHOOD, 'target_col': TARGET_COL,
        'fold': FOLD, 'year_start': YEAR_START, 'year_end': YEAR_END,
    }
    _save_run(name, model, losses_arr, preds_df, metrics, config)
    _upload_run_s3(name)
    _print_summary(metrics)

    del model
    jax.clear_caches(); gc.collect()
    return metrics


def train_eval_map(
    name: str,
    *,
    num_epochs: int,
    batch_size: int,
    ensemble_size: int,
    learning_rate: float,
    num_splits: int = 1,
    width: int = 256,
    depth: int = 2,
    seed: int = 0,
    seasonality_periods=None,
    num_seasonal_harmonics=None,
    feature_cols=None,
    interactions=None,
) -> dict:
    seasonality_periods    = seasonality_periods    or SEAS_DEFAULT['periods']
    num_seasonal_harmonics = num_seasonal_harmonics or SEAS_DEFAULT['harmonics']
    feature_cols           = feature_cols           or GEO_DEFAULTS['feature_cols']
    interactions           = interactions           or GEO_DEFAULTS['interactions']

    assert len(seasonality_periods) == len(num_seasonal_harmonics)
    assert feature_cols[0] == 'datetime'

    print(f'\n{"="*78}')
    print(f'  [MAP] {name}')
    print(f'        seasonality={seasonality_periods}/{num_seasonal_harmonics}')
    print(f'        features ({len(feature_cols)}): {feature_cols}')
    print(f'        width={width} depth={depth}  epochs={num_epochs}  batch={batch_size}')
    print(f'        ensemble={ensemble_size}  lr={learning_rate}  splits={num_splits}')
    print(f'{"="*78}')

    standardize_cols = [c for c in feature_cols if c != 'datetime']
    model = BayesianNeuralFieldMAP(
        width=width, depth=depth,
        freq='D', timetype='index',
        seasonality_periods=seasonality_periods,
        num_seasonal_harmonics=num_seasonal_harmonics,
        feature_cols=feature_cols,
        target_col=TARGET_COL,
        standardize=standardize_cols,
        observation_model=LIKELIHOOD,
        interactions=interactions,
    )

    keep = feature_cols + ['rainfall', 'rainfall_int', 'station_id']
    df_train_sub = df_train[keep]
    df_test_sub  = df_test [keep]

    stop, thr = _heartbeat_start(name)
    t0 = time.time()
    try:
        model = model.fit(
            df_train_sub,
            seed=jax.random.PRNGKey(seed),
            num_epochs=num_epochs,
            ensemble_size=ensemble_size,
            learning_rate=learning_rate,
            batch_size=batch_size,
            num_splits=num_splits,
        )
    finally:
        stop.set(); thr.join(timeout=1)
    fit_time_s = time.time() - t0
    print(f'  ✓ fit done in {fit_time_s:.0f}s ({fit_time_s/60:.1f} min)')

    losses_arr = np.asarray(model.losses_)
    _, _, metrics, preds_df = _build_predict_metrics(model, df_test_sub, name)
    metrics.update({
        'inference'             : 'map',
        'seasonality_periods'   : seasonality_periods,
        'num_seasonal_harmonics': num_seasonal_harmonics,
        'feature_cols'          : feature_cols,
        'interactions'          : interactions,
        'n_features'            : len(feature_cols),
        'fit_time_s'            : float(fit_time_s),
    })

    config = {
        'inference': 'map', 'name': name,
        'width': width, 'depth': depth,
        'num_epochs': num_epochs, 'batch_size': batch_size,
        'ensemble_size': ensemble_size, 'learning_rate': learning_rate,
        'num_splits': num_splits, 'seed': seed,
        'seasonality_periods': seasonality_periods,
        'num_seasonal_harmonics': num_seasonal_harmonics,
        'feature_cols': feature_cols, 'interactions': interactions,
        'likelihood': LIKELIHOOD, 'target_col': TARGET_COL,
        'fold': FOLD, 'year_start': YEAR_START, 'year_end': YEAR_END,
    }
    _save_run(name, model, losses_arr, preds_df, metrics, config)
    _upload_run_s3(name)
    _print_summary(metrics)

    del model
    jax.clear_caches(); gc.collect()
    return metrics


print('✓ helpers ready')
print('  GEO_DEFAULTS : n_features =', len(GEO_DEFAULTS['feature_cols']),
      ', n_interactions =', len(GEO_DEFAULTS['interactions']))
print('  SEAS_DEFAULT :', SEAS_DEFAULT)
print('  thresholds   : observed wet ≥', WET_THRESHOLD_MM, 'mm,  predicted wet ≥', PRED_DRY_THRESHOLD, 'mm')


## 7. Train — VI

Все гиперпараметры здесь. Меняй руками. Дефолты ниже — конфигурация, которая
успешно пробежала на 10 годах данных (CRPS ≈ 0.34 на полном test, Risk ≈ 10K в
зелёной зоне).


In [ ]:
m_vi = train_eval_vi(
    name='vi__WY_h1_10__ffrk_full',
    # core hyperparams
    num_epochs            = 100,
    batch_size            = 131072,
    ensemble_size         = 16,
    learning_rate         = 0.005,
    kl_weight             = 0.1,
    sample_size_posterior = 5,
    # architecture
    width = 256,
    depth = 2,
    seed  = 0,
    # seasonality / features / interactions: defaults из SEAS_DEFAULT / GEO_DEFAULTS
)


## 8. Train — MAP

MAP-инференс не использует `kl_weight` / `sample_size_posterior` (см. сигнатуру
`BayesianNeuralFieldMAP.fit`). Дефолтный lr в библиотеке = 0.005, дефолтные
epochs = 5000, но MAP **НЕ** скейлится по `train_data.shape[0] // batch_size`,
так что 5000 здесь — это реальные 5000 шагов на full-batch (либо `epochs * (n_train//batch)`
если `batch_size` задан).

Дефолты ниже — sensible starting point; пользователь скорректирует по риску.


In [ ]:
m_map = train_eval_map(
    name='map__WY_h1_10__ffrk_full',
    # core hyperparams
    num_epochs    = 2000,
    batch_size    = 131072,
    ensemble_size = 16,
    learning_rate = 0.005,
    num_splits    = 1,
    # architecture
    width = 256,
    depth = 2,
    seed  = 0,
)


## 9. Загрузка артефактов для аналитики

Все ячейки ниже самодостаточны — читают `results/bayesnf/runs/<EXP_NAME>/`.
По умолчанию смотрим VI-run; чтобы посмотреть MAP — поменяй `EXP_NAME` ниже на
`map__WY_h1_10__ffrk_full`.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EXP_NAME = 'vi__WY_h1_10__ffrk_full'   # ← поменять на 'map__...' чтобы посмотреть MAP
EXP_DIR  = Path('results/bayesnf/runs') / EXP_NAME
assert EXP_DIR.exists(), f'не найдена папка эксперимента: {EXP_DIR}'

metrics = json.loads((EXP_DIR / 'metrics.json').read_text())
config  = json.loads((EXP_DIR / 'config.json' ).read_text())
losses  = np.load   (EXP_DIR / 'losses.npy')
# pmap shape: (n_devices, ens_per_device, n_steps[, ...]) → схлопываем в (n_members, n_steps)
if losses.ndim > 2:
    losses = losses.reshape(-1, losses.shape[-1])
preds   = pd.read_parquet(EXP_DIR / 'preds.parquet')
preds['datetime'] = pd.to_datetime(preds['datetime'])

# координаты станций для пространственных диагностик
if 'df_test' not in globals():
    test_path = Path('results/bayesnf/data') / f'bayesnf_fold{config["fold"]}_test.parquet'
    df_test = pd.read_parquet(
        test_path,
        columns=['station_id','datetime','latitude','longitude','rainfall'],
        filters=[('datetime','>=', pd.Timestamp(f'{config["year_start"]}-01-01')),
                 ('datetime','<=', pd.Timestamp(f'{config["year_end"]}-12-31'))])
    df_test['datetime'] = pd.to_datetime(df_test['datetime'])

coords = (df_test[['station_id','latitude','longitude']]
            .drop_duplicates('station_id').set_index('station_id'))
preds = preds.merge(coords, left_on='station_id', right_index=True, how='left')

print(f'experiment        : {EXP_NAME}  (inference={config["inference"]})')
print(f'losses shape      : {losses.shape}  (ensemble × epochs)')
print(f'preds rows        : {len(preds):,}')
print(f'stations in test  : {preds["station_id"].nunique()}')
print(f'date range        : {preds["datetime"].min().date()} → {preds["datetime"].max().date()}')


## 10. Кривые потерь по ансамблю

Каждая голубая линия — один член ансамбля. Для VI это ELBO loss, для MAP —
negative log posterior. Согласованность ⇒ робастность; плато ⇒ запас по эпохам.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
mean_loss = losses.mean(0)
epochs = np.arange(losses.shape[1])
for ax, log in zip(axes, [False, True]):
    for m in range(losses.shape[0]):
        ax.plot(epochs, losses[m], color='tab:blue', alpha=0.3, lw=0.8)
    ax.plot(epochs, mean_loss, color='black', lw=1.6, label='ensemble mean')
    ax.fill_between(epochs, losses.min(0), losses.max(0),
                    color='tab:blue', alpha=0.15, label='min–max envelope')
    ax.set_xlabel('step'); ax.set_ylabel('loss')
    ax.grid(alpha=0.3); ax.legend()
    if log:
        ax.set_yscale('symlog'); ax.set_title('log scale')
    else:
        ax.set_title(f'{EXP_NAME} — {losses.shape[0]} members × {losses.shape[1]} steps')
plt.tight_layout(); plt.show()

final = mean_loss[-1]; tol = 0.01 * abs(final)
converged_at = next((e for e, v in enumerate(mean_loss) if abs(v - final) <= tol),
                    losses.shape[1] - 1)
print(f'≈ сошёлся на шаге {converged_at}/{losses.shape[1]-1} '
      f'(|loss − loss_final| ≤ 1%)')


## 11. Метрики предсказания

Три блока: **global** (на всём test), **wet-only** (observed ≥ 0.5 mm),
**dry-only** (observed < 0.5 mm). Дополнительный блок — runtime.


In [ ]:
def _fmt(k, v):
    if isinstance(v, float): return f'  {k:14s} = {v:>10.4f}'
    if isinstance(v, int):   return f'  {k:14s} = {v:>10,d}'
    return                          f'  {k:14s} = {v}'

ORDER_GLOBAL = ['rmse','mae','bias','crps','cov90','cov80','n_total','n_quantiles']
ORDER_WET    = ['rmse_wet','mae_wet','crps_wet','cov90_wet','cov80_wet','n_wet']
ORDER_DRY    = ['mae_dry','rmse_dry','bias_dry','n_dry']
ORDER_TIME   = ['fit_time_s','predict_time_s']

print('═'*42, '  GLOBAL  ', '═'*42)
for k in ORDER_GLOBAL: print(_fmt(k, metrics[k]))
print('\n' + '═'*42 + '  WET-ONLY' + '═'*42)
for k in ORDER_WET:    print(_fmt(k, metrics[k]))
print('\n' + '═'*42 + '  DRY-ONLY' + '═'*42)
for k in ORDER_DRY:    print(_fmt(k, metrics[k]))
print('\n' + '═'*42 + '  RUNTIME ' + '═'*42)
for k in ORDER_TIME:
    v = metrics[k]
    print(f'  {k:14s} = {v:>10.1f} s   ({v/60:.1f} min)')

print(f'\n  spec : seasonality={metrics["seasonality_periods"]} / '
      f'h={metrics["num_seasonal_harmonics"]}')
print(f'         features ({metrics["n_features"]}): {metrics["feature_cols"]}')


## 12. Wet-detector (двухэтапный диагностик)

Имитация Stage-1 у LGBM / индикаторного кригинга. Бинарная классификация
«мокро vs сухо»:

- **observed wet** $\Leftrightarrow$ `y_true ≥ 0.5 mm`
- **predicted wet** $\Leftrightarrow$ `mean_mm ≥ 0.4 mm`

Эти два порога — те же, что и в кригинге у тебя (P(wet) > 0.4 для классификации,
0.5 mm для определения сухости). Метрики:

| Метрика | Что значит |
|---------|------------|
| accuracy   | (TP + TN) / N |
| precision  | TP / (TP + FP) — насколько чисто «wet» когда модель так сказала |
| recall     | TP / (TP + FN) — сколько реальных wet модель не пропустила |
| F1         | гармоническое среднее P и R |
| Brier      | MSE между индикаторами (грубая prob calibration) |


In [ ]:
tp, fp, tn, fn = metrics['tp'], metrics['fp'], metrics['tn'], metrics['fn']
cm = np.array([[tn, fp], [fn, tp]])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# (1) confusion matrix
ax = axes[0]
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_xticklabels(['pred dry (<0.4)', 'pred wet (≥0.4)'])
ax.set_yticks([0, 1]); ax.set_yticklabels(['obs dry (<0.5)', 'obs wet (≥0.5)'])
ax.set_xlabel('predicted'); ax.set_ylabel('observed')
ax.set_title(f'Confusion matrix — {EXP_NAME}')
n_total = cm.sum()
for i in range(2):
    for j in range(2):
        v = cm[i, j]
        pct = 100 * v / n_total
        col = 'white' if v > cm.max()/2 else 'black'
        ax.text(j, i, f'{v:,}\n({pct:.1f}%)', ha='center', va='center', color=col, fontsize=11)
plt.colorbar(im, ax=ax, fraction=0.046)

# (2) bar chart of P/R/F1/Brier/Accuracy
ax = axes[1]
keys = ['accuracy', 'precision', 'recall', 'f1', 'brier']
vals = [metrics[k] for k in keys]
colors = ['tab:gray', 'tab:blue', 'tab:green', 'tab:purple', 'tab:red']
bars = ax.bar(keys, vals, color=colors, alpha=0.85)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v, f'{v:.3f}',
            ha='center', va='bottom', fontsize=10)
ax.set_ylim(0, 1.05)
ax.set_ylabel('score')
ax.set_title('Detector metrics (higher = better, except Brier)')
ax.grid(alpha=0.3, axis='y')

plt.tight_layout(); plt.show()

print(f'P(obs wet)  = {(tp+fn)/n_total:.3f}   (sanity ≈ 0.5–0.7 для среднеевропейского климата)')
print(f'P(pred wet) = {(tp+fp)/n_total:.3f}   (если сильно ниже obs — модель консервативна по wet)')


## 13. Калибровка интервалов — global vs wet-only

11-точечный reliability curve строим **дважды**: на всём test и только на
observed-wet. Глобальная over-coverage — типичный артефакт ZINB (нули
накачивают coverage низких α). Wet-only показывает реальную калибровку условно
на дождь.


In [ ]:
y    = preds['observed_mm'].to_numpy()
mean = preds['mean_mm'    ].to_numpy()
q05  = preds['q05'].to_numpy()
q95  = preds['q95'].to_numpy()
wet  = y >= WET_THRESHOLD_MM if 'WET_THRESHOLD_MM' in globals() else (y >= 0.5)

Q_LABELS = [5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 95]
Q_LEVELS = np.array([l / 100 for l in Q_LABELS])
q_cols = [f'q{l:02d}' for l in Q_LABELS]
Q = np.stack([preds[c].to_numpy() for c in q_cols], axis=1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# (1) reliability — global
emp_global = np.array([(y <= Q[:, k]).mean() for k in range(len(Q_LABELS))])
ax = axes[0]
ax.plot([0,1],[0,1],'--', color='grey', alpha=0.5, label='perfect')
ax.plot(Q_LEVELS, emp_global, 'o-', ms=6, color='tab:red', label='empirical')
ax.set_xlabel('nominal α'); ax.set_ylabel('empirical P(y ≤ qα)')
ax.set_title(f'Reliability — GLOBAL ({len(y):,} obs)')
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
ax.grid(alpha=0.3); ax.legend()

# (2) reliability — wet-only
emp_wet = np.array([(y[wet] <= Q[wet, k]).mean() for k in range(len(Q_LABELS))])
ax = axes[1]
ax.plot([0,1],[0,1],'--', color='grey', alpha=0.5, label='perfect')
ax.plot(Q_LEVELS, emp_wet, 'o-', ms=6, color='tab:green', label='empirical (wet)')
ax.set_xlabel('nominal α'); ax.set_ylabel('empirical P(y ≤ qα)')
ax.set_title(f'Reliability — WET-ONLY ({wet.sum():,} obs)')
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
ax.grid(alpha=0.3); ax.legend()

# (3) spread–skill (wet-only)
q50 = preds['q50'].to_numpy()
width = (q95 - q05)[wet]
err   = np.abs(q50 - y)[wet]
ax = axes[2]
K = 20
edges = np.quantile(width, np.linspace(0, 1, K+1))
centers = 0.5 * (edges[:-1] + edges[1:])
ids = np.clip(np.digitize(width, edges[1:-1]), 0, K-1)
mean_err = np.array([err[ids == k].mean() if (ids == k).any() else np.nan
                     for k in range(K)])
ax.plot(centers, mean_err, 'o-', color='tab:purple')
lim = max(np.nanmax(centers), np.nanmax(mean_err))
ax.plot([0, lim], [0, lim], '--', color='grey', alpha=0.5, label='y=x')
ax.set_xlabel('interval width  q95 − q05  (mm)')
ax.set_ylabel('|q50 − y|  (mm)')
ax.set_title(f'Spread–skill (wet, n={wet.sum():,})')
ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

print('Wet-only reliability (главное):')
for lvl, e in zip(Q_LEVELS, emp_wet):
    bias = e - lvl
    marker = '  ✓' if abs(bias) < 0.02 else ('  ↑' if bias > 0 else '  ↓')
    print(f'  α={lvl:.2f}  emp={e:.3f}  Δ={bias:+.3f}{marker}')


## 14. Observed vs Predicted — два режима

Слева **scatter wet-only** (потому что на dry просто облако в нуле), справа
**карта per-station MAE** на wet-точках.


In [ ]:
y    = preds['observed_mm'].to_numpy()
yhat = preds['mean_mm'    ].to_numpy()
wet  = y >= 0.5

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (1) scatter (wet-only)
ax = axes[0]
ax.hexbin(y[wet], yhat[wet], gridsize=60, mincnt=1, bins='log', cmap='viridis')
lim = max(y[wet].max(), yhat[wet].max())
ax.plot([0, lim], [0, lim], '--', color='red', alpha=0.6, lw=1.5)
ax.set_xlabel('observed (mm)'); ax.set_ylabel('predicted mean (mm)')
ax.set_title(f'Wet only ({wet.sum():,}), log-density')
ax.set_xlim(0, lim); ax.set_ylim(0, lim); ax.grid(alpha=0.3)

# (2) histogram of residuals (split wet/dry)
ax = axes[1]
res_wet = yhat[wet] - y[wet]
res_dry = yhat[~wet] - y[~wet]
qs = np.quantile(yhat - y, [0.005, 0.995])
ax.hist(res_dry, bins=60, range=qs, color='tab:gray',  alpha=0.6, label=f'dry (n={(~wet).sum():,})')
ax.hist(res_wet, bins=60, range=qs, color='tab:blue',  alpha=0.6, label=f'wet (n={wet.sum():,})')
ax.axvline(0, color='red', ls='--', alpha=0.6)
ax.set_xlabel('predicted − observed (mm)'); ax.set_ylabel('count')
ax.set_title('Residuals  by class')
ax.legend(); ax.grid(alpha=0.3, axis='y')

# (3) per-station MAE (wet-only) — карта
ax = axes[2]
df_wet = preds[wet].assign(abs_err=np.abs(yhat[wet] - y[wet]))
per_st = df_wet.groupby('station_id').agg(
    lat=('latitude','first'), lon=('longitude','first'),
    mae=('abs_err','mean'),   n=('abs_err','size'))
sc = ax.scatter(per_st['lon'], per_st['lat'], c=per_st['mae'],
                s=18, cmap='magma_r', edgecolor='black', linewidth=0.3)
plt.colorbar(sc, ax=ax, label='per-station MAE wet (mm)')
ax.set_xlabel('longitude (°)'); ax.set_ylabel('latitude (°)')
ax.set_title(f'Spatial MAE wet — {len(per_st)} test stations')
ax.grid(alpha=0.3)

plt.tight_layout(); plt.show()

print('top-5 worst stations (wet MAE):')
print(per_st.sort_values('mae', ascending=False).head(5).to_string())


## 15. Time-series по 4 примерным станциям

Чёрные точки — observed; красная — `mean_mm`; серая лента — `[q05, q95]`.
Берём worst-MAE, median, best и одну случайную.


In [ ]:
per_st = (preds.assign(abs_err=np.abs(preds['mean_mm'] - preds['observed_mm']))
                .groupby('station_id').agg(mae=('abs_err','mean')))
rng = np.random.default_rng(0)
picks = {
    'worst MAE' : per_st['mae'].idxmax(),
    'median MAE': per_st['mae'].sort_values().index[len(per_st)//2],
    'best MAE'  : per_st['mae'].idxmin(),
    'random'    : rng.choice(per_st.index),
}

fig, axes = plt.subplots(len(picks), 1,
                         figsize=(13, 2.4*len(picks)), sharex=True)
for ax, (label, sid) in zip(axes, picks.items()):
    s = preds[preds.station_id == sid].sort_values('datetime')
    ax.fill_between(s['datetime'], s['q05'], s['q95'],
                    color='lightgrey', label='95% PI')
    ax.plot(s['datetime'], s['mean_mm'], color='tab:red', lw=1.0,
            label='BayesNF mean')
    ax.scatter(s['datetime'], s['observed_mm'], color='black', s=6,
               label='observed', alpha=0.7)
    ax.set_ylabel('mm')
    ax.set_title(f'{label}: station={sid} (MAE={per_st.loc[sid,"mae"]:.2f} mm)')
    ax.grid(alpha=0.3)
axes[0].legend(loc='upper right', ncol=3)
plt.tight_layout(); plt.show()


## 16. Семивариограммный диагностик — wet days only

Saad et al. 2024, Fig. 5. По eq. 14:
$2\gamma(h, \tau) = \mathrm{Var}\bigl[Y(s+h,\,t+\tau) - Y(s,\,t)\bigr]$.

Фильтруем по дням где **≥10 % станций были мокрыми** (`observed_mm ≥ 0.5`) — на
полностью сухих днях γ ≈ 0 для obs и любые ≠0 для inferred ⇒ ratio
взрывается. На мокрых днях диагностика осмысленна.


In [ ]:
DIST_EDGES_KM = np.arange(0, 525, 50)
TIME_LAGS     = np.arange(0, 11)

stations = sorted(preds['station_id'].unique())
S = len(stations)

lat = np.array([coords.loc[s, 'latitude']  for s in stations])
lon = np.array([coords.loc[s, 'longitude'] for s in stations])
R = 6371.0
lat_r = np.radians(lat); lon_r = np.radians(lon)
dlat = lat_r[:, None] - lat_r[None, :]
dlon = lon_r[:, None] - lon_r[None, :]
a = np.sin(dlat/2)**2 + np.cos(lat_r)[:, None]*np.cos(lat_r)[None, :]*np.sin(dlon/2)**2
D_km = 2*R*np.arcsin(np.sqrt(np.clip(a, 0, 1)))

iu, ju = np.triu_indices(S, k=1)
d_pair = D_km[iu, ju]
bin_id = np.clip(np.digitize(d_pair, DIST_EDGES_KM[1:-1]), 0, len(DIST_EDGES_KM)-2)
nb = len(DIST_EDGES_KM) - 1

obs_panel  = (preds.pivot_table('observed_mm', index='datetime', columns='station_id')
                   .reindex(columns=stations).sort_index())
pred_panel = (preds.pivot_table('mean_mm',     index='datetime', columns='station_id')
                   .reindex(columns=stations).sort_index())

# фильтр: оставить дни где ≥10% станций имели y ≥ 0.5 mm
day_wet_frac = (obs_panel >= 0.5).mean(axis=1)
wet_days = day_wet_frac >= 0.10
print(f'wet days kept    : {wet_days.sum()} / {len(wet_days)} '
      f'({100*wet_days.mean():.1f}%)')

OBS  = obs_panel [wet_days].to_numpy(dtype=np.float32)
PRED = pred_panel[wet_days].to_numpy(dtype=np.float32)

def _semivariogram(panel):
    out = np.full((len(TIME_LAGS), nb), np.nan, dtype=np.float64)
    for k, tau in enumerate(TIME_LAGS):
        A = panel if tau == 0 else panel[:-tau]
        B = panel if tau == 0 else panel[tau:]
        diff = A[:, iu] - B[:, ju]
        sq = (diff * diff).astype(np.float64)
        mask = np.isfinite(sq)
        if not mask.any():
            continue
        flat_sq = sq[mask]
        flat_bin = np.broadcast_to(bin_id[None, :], sq.shape)[mask]
        sums = np.bincount(flat_bin, weights=flat_sq, minlength=nb)
        cnts = np.bincount(flat_bin, minlength=nb).astype(np.float64)
        with np.errstate(invalid='ignore'):
            out[k] = np.where(cnts > 0, 0.5 * sums / np.maximum(cnts, 1), np.nan)
    return out

print('computing empirical (wet days) ...', flush=True)
gamma_obs = _semivariogram(OBS)
print('computing inferred  (wet days) ...', flush=True)
gamma_inf = _semivariogram(PRED)

H = 0.5*(DIST_EDGES_KM[:-1] + DIST_EDGES_KM[1:])
HH, TT = np.meshgrid(H, TIME_LAGS)
fig = plt.figure(figsize=(14, 5))
for sub, (G, title) in enumerate(
        [(gamma_obs, 'Empirical γ̂ (wet days, observed)'),
         (gamma_inf, 'Inferred  γ̂ (wet days, BayesNF mean)')], 1):
    ax = fig.add_subplot(1, 2, sub, projection='3d')
    surf = ax.plot_surface(HH, TT, G, cmap='viridis',
                           edgecolor='k', linewidth=0.2, alpha=0.95)
    ax.set_xlabel('distance (km)'); ax.set_ylabel('time lag (days)')
    ax.set_zlabel('γ'); ax.set_title(title)
    fig.colorbar(surf, ax=ax, shrink=0.6, pad=0.1)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 6, figsize=(16, 5.5), sharex=True, sharey=True)
for k, tau in enumerate(TIME_LAGS):
    ax = axes.flat[k]
    ax.plot(H, gamma_obs[k], '-',  color='black',   lw=1.5, label='Empirical')
    ax.plot(H, gamma_inf[k], '--', color='tab:red', lw=1.5, label='Inferred')
    ax.set_title(f'τ = {tau} days')
    ax.grid(alpha=0.3)
    if k == 0: ax.legend()
    if k % 6 == 0: ax.set_ylabel('semivariance γ')
    if k // 6 == 1: ax.set_xlabel('distance (km)')
axes.flat[-1].axis('off')
plt.tight_layout(); plt.show()

with np.errstate(divide='ignore', invalid='ignore'):
    rel = np.abs(gamma_inf - gamma_obs) / np.where(gamma_obs > 0, gamma_obs, np.nan)
print(f'mean |γ_inf − γ_obs| / γ_obs over (h,τ)        = {np.nanmean(rel):.3f}')
print(f'  short lags τ ∈ {{0,1,2}}                       = {np.nanmean(rel[:3]):.3f}')
print(f'  long  lags τ ∈ {{3..10}}                       = {np.nanmean(rel[3:]):.3f}')
